# Lancio di un training con le nuove specs

In [1]:
from smash_rl import train
from smash_rl.runs import launch, list_runs, status, stop
import subprocess
import ipywidgets as widgets

## Debug pre run

In [ ]:
# lancio di pytest per verificare sia tutto a posto prima di addestrare
subprocess.run(["pytest", "-q", "--disable-warnings", "--tb=short"], check=True)

In [23]:
for run in list_runs():
    for key, value in status(run).items():
        print(f"{key}: {value}")
    print("\n")

run_id: 48
run_name: new_setup_a_b_second_attempt
pid: 1458754
pid_create_time: 1783942948.46
instance_base: 0
n_envs: 4
config_path: /home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt/config.json
log_path: /home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt/train.log
started_at: 2026-07-13T13:42:29
status: running
algo: dqn
total_steps: 3000000
alive: True
last_log_line: -----------------------------------


run_id: 49
run_name: new_setup_a_b_second_attempt_rv3
pid: 1459026
pid_create_time: 1783942962.33
instance_base: 4
n_envs: 4
config_path: /home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt_rv3/config.json
log_path: /home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt_rv3/train.log
started_at: 2026-07-13T13:42:43
status: running
algo: dqn
total_steps: 3000000
alive: True
last_log_line: -----------------------------------




## Run

In [6]:
# iperparametri di partenza per il training. Adattati da rl-zoo per atari, con qualche osservazione raccolta nei training preliminari

n_steps = int(3*1e6)   # facciamo 1.5 milioni di passi di addestramento
learning_rate = int(1e-4)    # per ora. Deve essere piccolo e verrà ottimizzato con optuna
buffer_size = int(0.2 * n_steps)  # 300k esperienze nel buffer, circa il 20% del totale, dovrebbe includere sia esperienze recenti che più vecchie senza privilegiare troppo nessuna delle due
learning_starts = round(0.03 * n_steps)  # iniziamo a imparare dopo il 3% dei passi, così da avere un buffer iniziale di esperienze
batch_size = 128    # aumentiamo la batch size per più stabilità nei gradient steps (il Q-learning tende a essere poco stabile)
gradient_steps = -1  # facciamo un gradient step per ogni passo di addestramento, così da sfruttare al massimo le esperienze raccolte
UTD = 0.25  # preso da rl-zoo per atari
train_freq = 4
n_envs = 4  # usiamo 4 ambienti in parallelo per raccogliere esperienze più velocemente
gradient_steps = round(UTD * n_envs * train_freq)  # calcoliamo il numero di gradient steps per avere un UTD di circa 0.25
target_update_interval = 10_000 # anche questo adattato da rl-zoo
exploration_initial_eps = 1.0  # iniziamo con eps=1.0 per esplorare molto all'inizio
exploration_final_eps = 0.02    # lasciamo un po' di esplorazione anche alla fine
exploration_fraction = 0.2      # l'esplorazione dura il 20% del training
gamma = 0.99    # questo dipende dal problema specifico, andrà adattato a parte


In [8]:
env_config = train.env_config(
    agent_char="MARIO",
    opp_char="LUIGI",
    stage="FINAL_DESTINATION",
    observation_function="full_obs",
    reward_function="v3",
    action_function="a_b",
    opp_level=1
)

dqn_config = train.DQN_config(
    exploration_initial_eps=exploration_initial_eps,
    exploration_final_eps=exploration_final_eps,
    exploration_fraction=exploration_fraction,
    buffer_size=buffer_size,
    learning_starts=learning_starts,
    batch_size=batch_size,
    gradient_steps=gradient_steps,
    target_update_interval=target_update_interval,
    gamma=gamma,
    train_freq=train_freq,
)

# Continua l'ultima run ripartendo dai suoi pesi finali. Con pretrained_path, run() ignora
# algo_kwargs (buffer/lr/...) e riprende gli iperparametri dal checkpoint: l'unica cosa che
# tocchiamo e' l'esplorazione. reset_exploration con initial==final tiene eps COSTANTE a
# exploration_final_eps, cosi' NON riparte la fase di esplorazione iniziale (eps da 1.0).
prev_run = "test_nuovo_setup_cpu3_full_actions_reward_v3"

train_config = train.TrainConfig(
    run_name="new_setup_a_b_second_attempt_rv3",   # nome nuovo: non sovrascrive i checkpoint della run finita
    algo="dqn",
    total_steps=n_steps,
    env_kwargs=env_config,
    algo_kwargs=dqn_config,
    n_envs=n_envs,
    pretrained_path=None,
    reset_exploration=None
)

launch(cfg=train_config)

new_setup_a_b_second_attempt_rv3: pid 1459026, istanze [4, 8), porte 51445-51448, log /home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt_rv3/train.log


RunHandle(run_name='new_setup_a_b_second_attempt_rv3', run_id=49, pid=1459026, log_path=PosixPath('/home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt_rv3/train.log'), config_path=PosixPath('/home/luca/Desktop/Code/smash/runs/new_setup_a_b_second_attempt_rv3/config.json'))

## Stop

In [ ]:
run_selector = widgets.Dropdown(
    options=[i["run_name"] for i in list_runs()],
    description='Select Run:',
    disabled=False,
)
display(run_selector)

In [ ]:
stop(run_selector.value)